In [1]:
# ============================================================
# 04_feature_engineering.py — Phase C Part 2: TF-IDF Features
# Internship Project: NLP Sentiment Analysis (IMDB 50K)
# ============================================================

import pandas as pd
import numpy as np
import os, joblib
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# ── Config ───────────────────────────────────────────────────
DATA_PATH  = "data/imdb_preprocessed.csv"
MODEL_DIR  = "outputs/models"
PLOT_DIR   = "outputs/plots"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PLOT_DIR,  exist_ok=True)

RANDOM_STATE = 42

def main():
    print("=" * 55)
    print("  Phase C Part 2 — Feature Engineering (TF-IDF)")
    print("=" * 55)

    # ── 1. Load ───────────────────────────────────────────────
    print("\n[1/5] Loading preprocessed data …")
    df = pd.read_csv(DATA_PATH)
    print(f"      {len(df):,} reviews loaded")

    X = df['processed_review'].fillna('')
    y = df['label']

    # ── 2. Train/Test Split ───────────────────────────────────
    print("\n[2/5] Train/Test Split (80/20, stratified) …")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    print(f"      Train: {len(X_train):,}  |  Test: {len(X_test):,}")
    print(f"      Train class dist: {y_train.value_counts().to_dict()}")
    print(f"      Test  class dist: {y_test.value_counts().to_dict()}")

    # Save splits (raw text) for reuse in model scripts
    pd.DataFrame({'text': X_train, 'label': y_train}).to_csv("data/train.csv", index=False)
    pd.DataFrame({'text': X_test,  'label': y_test }).to_csv("data/test.csv",  index=False)
    print("      Saved: data/train.csv, data/test.csv")

    # ── 3. TF-IDF Vectorizer ──────────────────────────────────
    print("\n[3/5] Fitting TF-IDF Vectorizer …")
    tfidf = TfidfVectorizer(
        max_features   = 50_000,      # vocab size
        ngram_range    = (1, 2),       # unigrams + bigrams
        min_df         = 2,            # ignore very rare terms
        max_df         = 0.95,         # ignore very common terms
        sublinear_tf   = True,         # log(tf) scaling
        strip_accents  = 'unicode',
        analyzer       = 'word',
    )
    X_train_tfidf = tfidf.fit_transform(X_train)
    X_test_tfidf  = tfidf.transform(X_test)

    feature_names = tfidf.get_feature_names_out()
    print(f"      Vocabulary size    : {len(feature_names):,} features")
    print(f"      Train matrix shape : {X_train_tfidf.shape}")
    print(f"      Test  matrix shape : {X_test_tfidf.shape}")
    print(f"      Matrix sparsity    : {1 - X_train_tfidf.nnz / np.prod(X_train_tfidf.shape):.4f}")

    # ── 4. Save Vectorizer & Matrices ─────────────────────────
    print("\n[4/5] Saving vectorizer & feature matrices …")
    joblib.dump(tfidf,           f"{MODEL_DIR}/tfidf_vectorizer.pkl")
    joblib.dump(X_train_tfidf,   f"{MODEL_DIR}/X_train_tfidf.pkl")
    joblib.dump(X_test_tfidf,    f"{MODEL_DIR}/X_test_tfidf.pkl")
    joblib.dump(y_train,         f"{MODEL_DIR}/y_train.pkl")
    joblib.dump(y_test,          f"{MODEL_DIR}/y_test.pkl")
    print(f"      Saved to {MODEL_DIR}/")

    # ── 5. Plot: Top TF-IDF Terms ─────────────────────────────
    print("\n[5/5] Plotting top TF-IDF features …")
    # Mean TF-IDF score per class
    pos_idx = y_train[y_train == 1].index
    neg_idx = y_train[y_train == 0].index

    # Map index to matrix row positions
    train_indices = list(X_train.index)
    pos_rows = [train_indices.index(i) for i in pos_idx if i in train_indices][:5000]
    neg_rows = [train_indices.index(i) for i in neg_idx if i in neg_idx][:5000]

    pos_mean = np.asarray(X_train_tfidf[pos_rows].mean(axis=0)).flatten()
    neg_mean = np.asarray(X_train_tfidf[neg_rows].mean(axis=0)).flatten()

    top_pos = np.argsort(pos_mean)[-15:][::-1]
    top_neg = np.argsort(neg_mean)[-15:][::-1]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for ax, idxs, color, title in [
        (axes[0], top_pos, '#22d3a5', 'Top 15 TF-IDF Terms — Positive'),
        (axes[1], top_neg, '#f87171', 'Top 15 TF-IDF Terms — Negative'),
    ]:
        vals   = pos_mean[idxs] if color == '#22d3a5' else neg_mean[idxs]
        labels = [feature_names[i] for i in idxs]
        ax.barh(range(15), vals[::-1], color=color, alpha=0.8)
        ax.set_yticks(range(15))
        ax.set_yticklabels(labels[::-1], fontsize=10)
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Mean TF-IDF Score')
        ax.spines[['top','right']].set_visible(False)

    plt.suptitle('TF-IDF Feature Importance by Sentiment Class', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{PLOT_DIR}/05_tfidf_top_features.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"      Saved: 05_tfidf_top_features.png")

    print("\n" + "="*55)
    print("  Feature Engineering Summary")
    print("="*55)
    print(f"  Vectorizer   : TF-IDF (unigrams + bigrams)")
    print(f"  Vocab size   : {len(feature_names):,} features")
    print(f"  Train shape  : {X_train_tfidf.shape}")
    print(f"  Test  shape  : {X_test_tfidf.shape}")
    print(f"  ngram_range  : (1, 2)")
    print(f"  max_features : 50,000")
    print(f"  sublinear_tf : True (log scaling)")
    print(f"\n✅ Phase C Part 2 complete!")

if __name__ == "__main__":
    main()


  Phase C Part 2 — Feature Engineering (TF-IDF)

[1/5] Loading preprocessed data …
      50,000 reviews loaded

[2/5] Train/Test Split (80/20, stratified) …
      Train: 40,000  |  Test: 10,000
      Train class dist: {1: 20000, 0: 20000}
      Test  class dist: {0: 5000, 1: 5000}
      Saved: data/train.csv, data/test.csv

[3/5] Fitting TF-IDF Vectorizer …
      Vocabulary size    : 50,000 features
      Train matrix shape : (40000, 50000)
      Test  matrix shape : (10000, 50000)
      Matrix sparsity    : 0.9976

[4/5] Saving vectorizer & feature matrices …
      Saved to outputs/models/

[5/5] Plotting top TF-IDF features …
      Saved: 05_tfidf_top_features.png

  Feature Engineering Summary
  Vectorizer   : TF-IDF (unigrams + bigrams)
  Vocab size   : 50,000 features
  Train shape  : (40000, 50000)
  Test  shape  : (10000, 50000)
  ngram_range  : (1, 2)
  max_features : 50,000
  sublinear_tf : True (log scaling)

✅ Phase C Part 2 complete!
